# பாடம் 18 (தொடர்ச்சி): *மனிதர்* செயலுக்கு அங்கீகாரம் அளித்ததை நிரூபிக்கும் ரசீதுகள்

இந்த பாடம் **எஜென்ட்** என்ன செய்தார் என்றும் **வாசல்** என்ன தீர்மானித்தது என்றும் நிரூபிக்கிறது. இந்த நோட்புக் மறைவான பாதியைச் சேர்க்கிறது: ஒரு **பெயரிடப்பட்ட மனிதர்** அந்த **சரியான** செயலுக்கு அனுமதி அளித்தது — முழுமையான அச்சிடப்பட்ட செயலின் மேலான தனித்த மனித கையொப்பம், ஆஃப்லைனில் சரிபார்க்கப்பட்டது.

இங்கு இரு கலைபொருள் வகைகளும் **பாடத்தின் ரசீதைப் போன்றதொரு ஓவியமாகே இருக்கின்றன**: ஒரு தளர்ந்த பயர்க்குள்ள பாத்திரம், `type` புலத்துடன், canonical JCS பைட்டுகளை நேரடியாக Ed25519 கையொப்பமிட்ட (கையொப்பப்பட்ட பைட்டுகளிலிருந்து விலக்கப்பட்ட) ஒரு கட்டமைக்கப்பட்ட `signature` பொருளுடன். அங்கீகார ரசீது ஒரு புதிய `type` (`human.approval.v1`), செயலின் வகையுடன் ஒன்று சேர்ந்து, ஒன்றுக்குள் `verify_chain` இரு ரசீது வகைகளையும் ஒரே குறியீட்டு பாதையைப் பயன்படுத்தி கண்டறியும். இந்த மனித-அங்கீகார ரசீது கல்வி நோக்கத்துக்காக பொது உருப்படியாக இதோ இங்கு வரையறுக்கப்பட்டது, draft-farley-acta-signed-receipts இல் வரையறுக்கப்படாத ரசீது வகை.

முதலில் உள்ள டெமோ சரிபார்ப்பாளரைவிட ஒருசில புதிய மேம்பாடுகள்: இங்கு ஸ்டார் செய்தியாக `signature.key_id` ஒரு **பழையவில் உறுதிப்படுத்தப்பட்ட விசை பதிவு** மீது தீர்மானிக்கப்படுகிறது, ரசீதின் உள்ளே உள்ள பொதுவான விசையை நம்புவதற்கு பதிலாக. பாடத்தின் சரிபார்ப்பு பட்டியலில் பரிந்துரைக்கப்படும் ( "சரிபார்ப்பு பொதுவான விசையை வெளியிடவும்"), அது போலவே போலியைத் தவிர்ப்பதற்குப் பதிலாக போலியில் மறுப்பு செய்யப்படுவதை உறுதி செய்கிறது.

இந்த நோட்புக் கற்றுக்கொள்ளும் விதி: **ஒரு கையொப்பமிடப்பட்ட அங்கீகாரம் தனக்குத் தனியா அதிகாரம் அல்ல.** அதிகாரம் இருக்கும் போது மட்டும் அங்கீகார ரசீதும் செயல் ரசீதும் ஒரே canonical செயலுடன் இணைப்பில் இருக்க வேண்டும், செயலாக்க நேரத்தில் தற்போதைய கொள்கை பதிப்பு, விசை மற்றும் காலாவதியாகாத எழுத்துப்புள்ளியில் இணைக்கப்பட்டிருக்க வேண்டும் மற்றும் அங்கீகாரம் ஏற்கப்படவில்லை. தோல்வி ஒன்றும் தனித்த காரணத்துடன் மறுக்கப்படுகிறது, ஆகவே *அதிகாரம் பழகு போனது* மற்றும் *செயலை மாற்றப்பட்டது* ஆகியவை வேறுபடுத்த முடியும்.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## துல்லியமான செயல்

அங்கீகார நுணுக்கம் என்பது **கானோனிக்கல் செயல் பொருள்** — "இடைக்காலத்தை ஒப்புதல்" போன்ற அபாஷ் குறியீடுகள் அல்ல, ஆனால் துல்லியமாகக் குறிப்பிடப்பட்ட, முழுமையாக வரையறுக்கப்பட்ட செயல். முழு பொருளை கையொப்பமிடும் போது (அதில் இருந்து ஒரு குறும்படம் உருவாக்கி) அதை ஒருவரது மனிதன் பார்த்துக்கொண்டார் என்றும் வேறெதை பார்த்திருக்கவில்லை என்றும் பின்னர் நிரூபிக்க நாம் இதை பயன்படுத்துகிறோம்.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ஒரு மூடி, இரண்டு அதிகாரிகள்

ஒவ்வொரு ரசீது பாடத்தின் மூடியே: ஒரு சமமான தரவு `type` புலத்துடன், கூடவே `signature` பொருள் (`alg`, `sig`, `key_id`) இது கையொப்பமிடப்பட்ட பைட்டுகளின் பகுதி அல்ல. `verify_envelope` இரு ரசீதுகளுக்குமான பகிரப்பட்ட கட்டமைப்பு + கையொப்பக்கான உறுதிப்படுத்தல்; அது எது **பின்ரிக்கப்பட்ட திறவுகோல் பதிவியல்** `signature.key_id` ஐ எதிர்கொள்கிறது என்பதுதான் அதிகாரிகளை தனித்துவம் செய்யிறது:

- **அங்கீகார ரசீது** (`human.approval.v1`) — பெயரிடப்பட்ட அங்கீகாரி, முழுமையான நிரூபிக்கப்பட்ட செயல்பாடு **மற்றும் அதன் சுருக்கம்**, `policy_version`, வெளியீடு + கடைசித்திகதி நேரக்கோற்றுக்கள். ஒருமுறை பயன்படுத்தல் சங்கிலி மட்டத்தில் கண்காணிக்கப்படுகிறது.
- **செயல் ரசீது** (`agent.action.v1`) — முகவர் அடையாளம், `run_id`, அதே நிரூபிக்கப்பட்ட செயலின் **சுருக்கம்**, நிறைவேற்றல் முடிவு + நேரக்கோடு, மற்றும் `parent_approval_ref`: அங்கீகார ரசீதின் `receipt_hash`, பாடத்தின் சங்கிலியில் உள்ள `previous_receipt_hash` என்ற நடைமுறைபோல.

பகிரப்பட்ட `action_digest` புலம் இணைப்பை சார்ந்தது. `key_id` கையொப்ப பொருளில் lookup விளக்கமாக மட்டுமே இருக்கிறது: அதை வேறு பின்ரிக்கப்பட்ட திறவுகோலில் மாற்றுதல் கையொப்ப சோதனையை தோல்வி அடையச் செய்யும், ஆகையால் அது எந்தவொரு நீட்சியையும் தராது.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: இணைப்பு உண்மையில் முடிவடையும் இடம்  

`verify_chain` என்பது இரண்டு கையொப்ப சோதனைகளுக்கு மேலான ஒரு வசதிப்பயனாளர் பூட்டி değildir. இது பகிரப்பட்ட canonical `action_digest`, அங்கீகாரத்தின் கொள்கை/சாவி/காலாவதி **புதியமைவு**, மற்றும் அந்த அங்கீகாரத்தின் **ஒரு முறை மட்டுமே பயன்படுத்தப்படுத்தல்** ஆகியவை ஒன்றாகவே, ஒருநிலையில் செயல் இயக்கப்படுகின்ற போது சரிபார்க்கப்படும் ஒரே இடம் ஆகும்.  

ஒவ்வொரு தோல்வியும் **தனித்துவமான காரணத்துடன்** மறுக்கப்படுகிறது, எனவே மறுப்பை வாசிப்பவர் அதிகாரம் காலாவதி ஆனதா (கொள்கை நகர்த்தப்பட்டது, சாவி மாற்றப்பட்டது, அங்கீகாரம் காலாவதி ஆனது, அங்கீகாரம் பயன்படுத்தப்பட்டது) அல்லது இன்னும் செல்லுபடியாகும் அங்கீகாரத்தின் கீழ் செயல்படுத்தப்பட்ட செயல் மாறி விட்டதா (டைகெஸ்ட் மாற்றம்) என்பதை அறிய முடியும்.  


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## தொடர்பு எதை பிடிக்கிறது

கீழே உள்ள ஒவ்வொரு வழியும் **தொகுக்கப்பட்ட** நிலையில் **வேறுபட்ட காரணத்துடன்** தோல்வி கொள்கின்றன. முதலாவது தொகுப்பு சாமானியமாக உள்ளவை (தனியுரிமை மீறல், குழப்பப்பட்ட பிரதிநிதி, மீண்டும் விளையாடு, அதிகாரத்தின் மீது போலி செயல், தவறான உள்ளீடு). இரண்டாவது தொகுப்பு அந்த சொத்துக்களை உறுதிப்படுத்தும் பூர்வீகத்தை உண்மையானதாக மாற்றுகிறது:

- **பழைய அதிகாரம்** — கையெழுத்து இன்னும் செல்லுபடியானது, ஆனால் கொள்கை பதிப்பு மாற்றம் ஆகிவிட்டது, ஏற்புடைய திறவுகோல் நிரந்தர பதிவிலிருந்து நீக்கப்பட்டது, அல்லது அனுமதி நடைமுறைக்கு முன் காலாவதியாகி விட்டது;
- **தொகுப்பு மாற்று** — செல்லுபடியான கையெழுத்து செய்யப்பட்ட செயல் ரசீது, அதில் உள்ள `parent_approval_ref` *உண்மையான* அனுமதியைக் குறிக்கிறது, ஆனால் அந்த அனுமதியின் இயல்புநிலை செயல் தொகுப்பு செயல் முறையில் நடைமுறைப்படுத்தப்படும் செயல் தொகுப்புடன் overeenௌளவில்லை.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## இது என்ன நிரூபிக்கிறது — மற்றும் என்ன அல்ல

**நிரூபிக்கிறது:** ஒரு பெயரிடப்பட்ட மனிதன் *இந்தச் சரியான கனானிக்கல் நடவடிக்கையினை* அங்கீகரித்துள்ளார் (முழு நடவடிக்கை + சுருக்கம், பின்முடிக்கப்பட்ட பதிவு மூலம் தீர்க்கப்பட்ட ஒரு விசையால் கையொப்பமிடப்பட்டது), மற்றும் முகவர் *அதே அங்கீகரிக்கப்பட்ட நடவடிக்கையை துல்லியமாக* (அதே சுருக்கம், அங்கீகாரம் `receipt_hash` மூலம் கட்டுப்படுத்தப்படும், பாடத்தின் சொந்த சங்கிலி நடைமுறை) ஒரே ஒரு முறையே செயல்படுத்தினார். இரண்டு பக்கங்களும் மாற்றப்பட்டால், சங்கிலி மூடப்போகும், மற்றும் நிராகரிப்பு காரணம் **எந்த** பண்புக்கே கடி அடித்தது என்று உங்களுக்கு தெரிவிக்கும்: காலாவதியான அதிகாரம் எதிரான மாற்றப்பட்ட நடவடிக்கை.

**நிரூபிக்காது:** அங்கீகாரம் UI மனிதன் கையொப்பமிட்டதுதான் என்று அவர்கள் நினைத்ததை காட்டியது (WYSIWYS என்பது தன் பிரச்சினை), விசை மாற்றத்திற்கு முன் வேடிக்கைபடுத்தப்படவில்லை அல்லது திருடப்படவில்லை, அல்லது அடுத்தடுத்த விளைவுகள் நடவடிக்கையுடன் பொருந்தின. கையொப்பமிடப்பட்டது என்பது அங்கீகாரம் அல்ல: காலாவதியான கொள்கைக்குள் செல்லுபடியான கையொப்பம், மாற்றப்பட்ட விசை, காலாவதி காலம், அல்லது வேறு சுருக்கம் இங்கு எந்தவிதமும் வழங்காது.

இரண்டு ரசீது வகைகள் பாடத்தின் தொப்பியை மற்றும் ஒரே `verify_chain` குறியீட்டு பாதையை வழக்கமாகப் பகிர்கின்றன: முக்கிய குறிப்பேடு நடவடிக்கை ரசீதுகளுக்கான கட்டமைப்பானது மனித அங்கீகாரத்தைச் சரிபார்க்கும் அதே குறியீட்டும் ஆகும். ஒரு சரிபார்ப்பான் ஒப்பந்தம், தனித்துவமான பின்முடிக்கப்பட்ட அதிகாரங்கள், கனானிக்கல் நடவடிக்கை சுருக்கம் மற்றும் வேறு எதுவுமில்லாமல் இணைக்கப்பட்டவை.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
